# Bathymetric adjustment of rating curves (BARC and BARC-SS) with FIMserv_BARC-SS

OWP HAND-FIM rating curves miss the river channel below the water surface because the DEM does not capture it. This notebook adds that channel geometry to the hydrotables before running HAND-FIM:

- **BARC**: adds the bankfull channel area and wetted perimeter to the hydrotable and recomputes discharge with Manning's equation.
- **BARC-SS**: BARC, plus shifts each rating curve's stage up by the reach's mean-flow depth and anchors it at zero flow.

Channel width and depth come from [HydroGeom](https://github.com/Reizrb/HydroGeom). The example uses **HUC8 11110203** (Arkansas River), a **trapezoidal channel with a = 0.4**, and the flood of **3 January 2016**.

[FIMserv_BARC-SS](https://github.com/Reizrb/FIMserv_BARC-SS) is [FIMserv](https://github.com/sdmlua/FIMserv) with BARC and BARC-SS added as one extra step (`fm.BathymetricAdjustment`).

Run this notebook from an empty folder. FIMserv creates `code/`, `data/`, and `output/` there.

### STEP 1. Install and import

In [ ]:
# !pip install "git+https://github.com/Reizrb/FIMserv_BARC-SS" "git+https://github.com/Reizrb/HydroGeom"
import fimserve as fm

### STEP 2. Settings

- `method`: `"BARC"` or `"BARC-SS"`
- `shape` and `value`: the channel shape used to compute bankfull area and wetted perimeter
  - `"a"`: trapezoid, `value` = bottom width / top width (0 = triangle, 1 = rectangle)
  - `"r"`: Dingman's r, `value` = r (1 = triangle, 2 = parabola, larger = rectangle)
  - `"z"`: side slope, `value` = z (z:1)

In [ ]:
huc = "11110203"
method = "BARC-SS"
shape, value = "a", 0.4

start_date = "2015-12-26"
end_date = "2016-01-05"
value_times = ["2016-01-03 00:00:00"]

### STEP 3. Download OWP HAND-FIM (version 4.5)

In [ ]:
fm.DownloadHUC8(huc, version="4.5")

### STEP 4. Apply BARC or BARC-SS to the hydrotables

This gets the channel geometry for every reach in the HUC8 and rewrites the hydrotables. The originals are saved in `output/flood_11110203/original_hydrotables/`, so you can run this again with other settings. `fm.RestoreHydrotable(huc)` puts the originals back.

In [ ]:
fm.BathymetricAdjustment(huc, method=method, shape=shape, value=value)

### STEP 5. Compare a rating curve before and after

In [ ]:
fm.plotBathymetricSRC(huc)          # or fm.plotBathymetricSRC(huc, hydroid=...)

### STEP 6. Get NWM discharge and run HAND-FIM

In [ ]:
fm.getNWMretrospectivedata(huc, start_date, end_date, value_times=value_times)
fm.runOWPHANDFIM(huc)

### STEP 7. Map the flooded area

In [ ]:
import glob
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from matplotlib.colors import ListedColormap

fim = sorted(glob.glob(f"output/flood_{huc}/{huc}_inundation/*_inundation.tif"))[-1]
with rasterio.open(fim) as r:
    flooded = r.read(1) > 0
    b, crs, res = r.bounds, r.crs, r.res

fig, ax = plt.subplots(figsize=(9, 7))
ax.imshow(np.ma.masked_where(~flooded, flooded), cmap=ListedColormap(["#1f78b4"]),
          extent=(b.left, b.right, b.bottom, b.top), interpolation="nearest")
gpd.read_file(f"output/flood_{huc}/{huc}/wbd8_clp.gpkg").to_crs(crs).boundary.plot(ax=ax, color="0.4", lw=0.8)
ax.set_title(f"{method} ({shape} = {value}), HUC8 {huc}: {flooded.sum() * res[0] * res[1] / 1e6:,.1f} km² flooded")
ax.set_axis_off()
plt.savefig(f"FIM_{huc}_{method}_{shape}{value}.png", dpi=200, bbox_inches="tight")
plt.show()

### Run again with another method or shape

FIMserv skips HAND-FIM if a flood map for the same discharge already exists, so rename the current map first. Then change the settings in STEP 2 and run STEPS 4 to 7 again.

In [ ]:
import os
os.rename(fim, fim.replace("_inundation.tif", f"_{method}_{shape}{value}_inundation.tif"))

To map the original HAND-FIM for comparison, run `fm.RestoreHydrotable(huc)` instead of STEP 4.

A benchmark flood map for this event (NOAA aerial imagery, 2016-01-03) is available in [FIMbench](https://fimbench.tethys.ciroh.org/).